# ♟️ ChessMarro Training Pipeline

This notebook provides a complete training pipeline for the ChessMarro chess AI model. All cells have been converted to functions, and there's an interactive configuration form at the end to select your dataset, model name, and base model.

## Usage:
1. Run all cells from top to bottom
2. Scroll to the bottom and run the last cell for the interactive training form
3. Choose your dataset, model name, and optionally load a base model to continue training

## Key Features:
- ✅ Modular function-based architecture
- 📊 Support for parquet datasets
- 🔧 Checkpoint recovery (resume from last epoch)
- 💾 Automatic model saving
- 🧪 Model quality testing

In [1]:
#!pip install chess

In [2]:
import random
import os
from torch.utils.data import Dataset, DataLoader
import chess
import chess.svg
from IPython.display import display, SVG
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import pyarrow as pa 
import pyarrow.parquet as pq
import torch
import numpy as np
import math

## 1️⃣ Setup & Imports
All necessary modules and custom chess library imports are set up below.

In [3]:
def read_data(file, page, size):
    """
    Read a batch from parquet file
    
    Args:
        file: Path to parquet file
        page: Page number to read (0-indexed)
        size: Batch size for each page
    
    Returns:
        DataFrame with the requested page data
    """
    try:
        with pq.ParquetFile(file) as pf:
            total_pages = math.ceil(pf.metadata.num_rows / size)
            
            # Validar que la página existe
            if page >= total_pages:
                print(f"⚠️ Page {page} exceeds available pages ({total_pages}). Reading last page instead.")
                page = total_pages - 1
            
            if page < 0:
                page = 0
            
            iterb = pf.iter_batches(batch_size=size)
            
            # Saltarse las páginas anteriores
            for i in range(page):
                try:
                    next(iterb)
                except StopIteration:
                    print(f"⚠️ Reached end of file at page {i}. Returning last available page.")
                    break
            
            # Leer la página requerida
            try:
                batches = next(iterb)
            except StopIteration:
                print(f"❌ Could not read page {page}. File may be corrupted or too small.")
                raise ValueError(f"Page {page} not available in {file}")
            
            df_chess = pa.Table.from_batches([batches]).to_pandas()
            
            batches = None
            iterb = None
            
            # Validar que tenemos datos
            if df_chess is None or len(df_chess) == 0:
                raise ValueError(f"No data found in page {page}")
            
            # Reshape board column si existe
            if 'board' in df_chess.columns:
                df_chess['board'] = df_chess['board'].apply(lambda board: board.reshape(77, 8, 8).astype(int))
            
            print(f"✅ Read {len(df_chess)} rows from page {page}")
            return df_chess
            
    except Exception as e:
        print(f"❌ Error reading data: {type(e).__name__}: {e}")
        raise

def get_total_pages(file_path, size):
    """Get total number of pages in parquet file"""
    try:
        with pq.ParquetFile(file_path) as pf:
            total_rows = pf.metadata.num_rows
            total_pages = math.ceil(total_rows / size)
            return total_pages
    except Exception as e:
        print(f"❌ Error getting total pages: {e}")
        return 1  # Default to 1 if error

## 2️⃣ Data Loading Functions
Functions to read and process chess datasets from parquet files.

In [4]:
def test_reading(file_path='lc0_converted.parquet_v2_suffled.gzip', page=0, size=300):
    df_chess = read_data(file_path, page, size)
    board = chess.Board(df_chess.loc[100, 'fen_original'])

    # Mostrar el tablero inicial
    svg_board = chess.svg.board(board=board, size=300)
    display(SVG(svg_board))
    print(df_chess.loc[100])
    print(df_chess.loc[100, 'fen_original'])

In [5]:
def create_dataLoaders(df_chess):    
    # This class is used by pytorch for providing data to model. We use it to read dataset and convert to desired format 
    class ChessDataset(Dataset):
        def __init__(self, df):
            self.dataframe = df

        def __len__(self):
            return len(self.dataframe)
        
        def __getitem__(self, idx):
            fen = torch.tensor(self.dataframe.loc[idx, 'board'], dtype=torch.float32)
            uci_best_move = self.dataframe.loc[idx, 'best']
            value = self.dataframe.loc[idx, 'value']
            return (fen, uci_best_move, value, self.dataframe.loc[idx, 'fen_original'])    

    # Validar dataset
    if df_chess is None or len(df_chess) == 0:
        raise ValueError("❌ Dataset is empty or None. Cannot create dataloaders.")
    
    required_cols = {'board', 'best', 'value', 'fen_original'}
    missing_cols = required_cols - set(df_chess.columns)
    if missing_cols:
        raise ValueError(f"❌ Dataset missing required columns: {missing_cols}")

    # We can select a small sample to try the training algorithm
    # In this case we select all the Dataset provided
    df_selected = df_chess.sample(n=len(df_chess), random_state=42).reset_index(drop=True)

    # Then we create an Object with this dataset
    data_train = ChessDataset(df_selected)

    # Handle tiny datasets so train/test and dataloaders are never empty
    dataset_size = len(data_train)
    print(f"📊 Dataset size: {dataset_size} samples")
    
    if dataset_size < 2:
        raise ValueError(f"❌ Dataset too small: {dataset_size} rows. Need at least 2 rows to split train/test.")

    # Select 80% for training and 20% for testing, minimum 1 sample each
    train_size = max(1, int(0.8 * dataset_size))
    test_size = dataset_size - train_size
    if test_size == 0:
        test_size = 1
        train_size = dataset_size - 1
    
    print(f"   Train: {train_size} | Test: {test_size}")

    train_dataset, test_dataset = torch.utils.data.random_split(data_train, [train_size, test_size])

    # Create the Dataloader
    # drop_last=False prevents empty loaders when split size < batch_size
    batch_size = min(64, max(1, train_size // 2))  # Scale batch size to dataset
    dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    dataloader_test = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    
    if len(dataloader) == 0 or len(dataloader_test) == 0:
        raise ValueError("❌ Empty dataloader created. Dataset split or batch size issue.")
    
    return (dataloader, dataloader_test)
    
def test_dataloader(file_path='lc0_converted.parquet.gzip', page=0, size=300):
    df_chess = read_data(file_path, page, size)
    (dataloader, dataloader_test) = create_dataLoaders(df_chess)
    for batch in dataloader:
        print("\nBest Move Matrix:")
        print(batch[1])
        break  # Imprimir the first to show only converted data

## 3️⃣ Model Architecture
Neural network definition with residual connections for chess position evaluation.

In [6]:
class Mish(nn.Module):
    """Activación Mish: x * tanh(softplus(x))"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block para atención de canales"""
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResBlock(nn.Module):
    """Bloque Residual con Pre-activación y SE"""
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.se = SEBlock(channels)
        self.activation = Mish()

    def forward(self, x):
        residual = x
        out = self.bn1(x)
        out = self.activation(out)
        out = self.conv1(out)
        
        out = self.bn2(out)
        out = self.activation(out)
        out = self.conv2(out)
        
        out = self.se(out)
        return out + residual

class ChessNetPV_Optimized(nn.Module):
    def __init__(self, num_blocks=12): # 6 o 12 bloques es mucho más profundo que el original
        super(ChessNetPV_Optimized, self).__init__()

        in_channels = 77
        base_channels = 256 # Ancho constante profesional 
        head_bottleneck_channels = 32 # Para reducir parámetros en FC [6]
        

        # Entrada inicial
        self.conv_input = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1, bias=False)
        
        # Torre Residual (Cuerpo de la red)
        self.res_tower = nn.Sequential(
            *[ResBlock(base_channels) for _ in range(num_blocks)]
        )
        
        # BN y Activación final de la torre (por arquitectura de pre-activación)
        self.final_bn = nn.BatchNorm2d(base_channels)
        self.final_act = Mish()

        # --- CABEZAL DE POLÍTICA (4096 salidas) ---
        self.policy_conv = nn.Conv2d(base_channels, head_bottleneck_channels, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(head_bottleneck_channels)
        self.policy_fc = nn.Linear(head_bottleneck_channels * 8 * 8, 4096)

        # --- CABEZAL DE VALOR (1 salida) ---
        self.value_conv = nn.Conv2d(base_channels, head_bottleneck_channels, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(head_bottleneck_channels)
        self.value_fc1 = nn.Linear(head_bottleneck_channels * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        # Cuerpo
        x = self.conv_input(x)
        x = self.res_tower(x)
        x = self.final_act(self.final_bn(x))

        # Política
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = self.policy_fc(p)

        # Valor
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        value = F.relu(self.value_fc1(v))
        value = self.tanh(self.value_fc2(value))

        return policy, value

In [7]:
def initialize_model(base_model_path=None):
    """Initialize the model and device"""
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using {device} device")
    model = ChessNetPV_Optimized().to(device)
    
    # Load base model if provided
    if base_model_path and os.path.exists(base_model_path):
        model.load_state_dict(torch.load(base_model_path, map_location=device))
        print(f"✅ Loaded base model from {base_model_path}")
    
    return device, model

## 4️⃣ Model Training
Training functions with early stopping, loss calculation, and checkpoint management.

In [8]:
def train(model, dataloader, dataloader_test, device):
    max_epoch = 10 
    log_interval = 300
    early_stopping_patience = 5 
    best_loss = float('inf')
    best_accuracy = 0
    patience = 0 

    learning_rate = 0.005 
    # Criterios para las dos cabezas
    criterion_policy = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_value = nn.MSELoss() 
    
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  

    # Validación exhaustiva de dataloaders
    if dataloader is None or dataloader_test is None:
        raise ValueError("❌ Dataloaders are None")
    
    if len(dataloader) == 0:
        raise ValueError("❌ Training dataloader is empty")
    
    if len(dataloader_test) == 0:
        raise ValueError("❌ Test dataloader is empty")
    
    print(f"✅ Train batches: {len(dataloader)} | Test batches: {len(dataloader_test)}")

    for epoch in range(max_epoch):
        try:
            # --- FASE DE ENTRENAMIENTO ---
            model.train()  
            pbar = tqdm(total=len(dataloader), desc=f'Training Epoch {epoch}')
            total_train_loss = 0
            batch_count = 0

            # Nota: El dataloader debe devolver (data, target_policy, target_value)
            for batch_idx, batch_data in enumerate(dataloader):
                try:
                    data, target_p, target_v, fen_original = batch_data
                    data = data.to(device)
                    target_p = target_p.to(device) # El movimiento (0-4095)
                    target_v = target_v.to(device).float() # El valor de lc0 (-1 a 1)

                    optimizer.zero_grad()
                    
                    # La red devuelve dos salidas
                    policy_out, value_out = model(data)
                    
                    # Calculamos las pérdidas por separado
                    loss_p = criterion_policy(policy_out, target_p)
                    # Reshape both tensors to 1D to avoid dimension mismatches
                    loss_v = criterion_value(value_out.view(-1), target_v.view(-1))
                    
                    # Pérdida total combinada
                    loss = loss_p + (2 * loss_v)
                    
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
                    optimizer.step()

                    total_train_loss += loss.item()
                    batch_count += 1
                    if batch_idx % log_interval == 0:
                        pbar.set_postfix(L_pol=loss_p.item(), L_val=loss_v.item())
                    pbar.update(1)
                    
                except Exception as e:
                    print(f"\n⚠️ Error in training batch {batch_idx}: {e}")
                    pbar.close()
                    raise

            avg_train_loss = total_train_loss / max(batch_count, 1)
            pbar.close()
            print(f"   Avg Train Loss: {avg_train_loss:.4f}")

            # --- FASE DE TEST ---
            model.eval()
            test_loss = 0
            correct_policy = 0
            total_mse_v = 0
            test_batch_count = 0

            with torch.no_grad():
                for test_data in dataloader_test:
                    try:
                        data, target_p, target_v, fen_original = test_data
                        data = data.to(device)
                        target_p = target_p.to(device)
                        target_v = target_v.to(device).float()

                        policy_out, value_out = model(data)
                        
                        # Pérdida en test
                        l_p = criterion_policy(policy_out, target_p)
                        # Reshape both tensors to 1D to avoid dimension mismatches
                        l_v = criterion_value(value_out.view(-1), target_v.view(-1))
                        test_loss += (l_p + l_v).item()
                        
                        # Métrica de precisión para la política (movimientos)
                        pred = policy_out.argmax(dim=1, keepdim=True) 
                        correct_policy += pred.eq(target_p.view_as(pred)).sum().item()
                        
                        # Métrica de error para el valor
                        total_mse_v += l_v.item()
                        test_batch_count += 1
                        
                    except Exception as e:
                        print(f"\n⚠️ Error in test batch: {e}")
                        raise

            test_loss /= max(test_batch_count, 1)
            test_accuracy = 100. * correct_policy / max(len(dataloader_test.dataset), 1)
            avg_v_error = total_mse_v / max(test_batch_count, 1)

            print(f'🔎 Test Loss: {test_loss:.4f} | Acc: {test_accuracy:.2f}% | MSE: {avg_v_error:.4f}')

            # Early stopping basado en la pérdida total de test
            if test_loss < best_loss:
                best_loss = test_loss
                best_accuracy = test_accuracy
                patience = 0 
            else:
                patience += 1
                if patience > early_stopping_patience:
                    print('⚠️ Early stopping triggered (no improvement).')
                    break

            scheduler.step()
            
        except Exception as e:
            print(f"❌ Error in epoch {epoch}: {type(e).__name__}: {e}")
            raise

    print(f'🎯 Training completed. Best accuracy: {best_accuracy:.2f}%')

In [9]:
def start_training_loop(model, device, dataset_file, model_name, start_epoch=0, num_epochs=7):
    """
    Training loop that iterates through dataset pages and trains the model
    
    Args:
        model: The neural network model
        device: torch device (cuda, mps, or cpu)
        dataset_file: Path to the parquet dataset file
        model_name: Name prefix for saved model checkpoints (without extension)
        start_epoch: Starting epoch number (default 0)



                num_epochs: Number of NEW epochs to train from start_epoch (default 7)
    """
    # Validar dataset existe
    if not os.path.exists(dataset_file):
        raise FileNotFoundError(f"❌ Dataset file not found: {dataset_file}")
    
    print(f"✅ Dataset found: {dataset_file}")
    print(f"📈 Training: {num_epochs} new epochs, starting from epoch {start_epoch}")
    
    # --- Lógica de Recuperación ---
    actual_start_epoch = start_epoch
    # Buscamos si existen modelos guardados para continuar desde el último
    checkpoints = [f for f in os.listdir() if f.startswith(model_name) and f.endswith('.pth')]
    
    if checkpoints:
        # Extraemos los números de las épocas y buscamos el máximo
        epochs_nums = []
        for f in checkpoints:
            try:
                epoch_num = int(f.replace(model_name, '').replace('_epoch', '').replace('.pth', ''))
                epochs_nums.append(epoch_num)
            except ValueError:
                pass
        
        if epochs_nums:
            last_epoch = max(epochs_nums)
            model_path = f'{model_name}_epoch{last_epoch}.pth'
            
            try:
                model.load_state_dict(torch.load(model_path, map_location=device))
                actual_start_epoch = last_epoch + 1
                print(f"🚀 Checkpoint recovered: {model_path}")
                print(f"   Resuming training from epoch {actual_start_epoch}")
            except Exception as e:
                print(f"⚠️ Could not load checkpoint {model_path}: {e}")
                print(f"   Starting fresh from epoch {start_epoch}")
                actual_start_epoch = start_epoch

    if not checkpoints:
        print("🆕 No checkpoints found. Starting training from scratch.")

    # --- Bucle de entrenamiento modificado ---
    # Calcular rango final: desde actual_start_epoch hasta actual_start_epoch + num_epochs
    final_epoch = actual_start_epoch + num_epochs
    failed_epochs = []
    epoch_count = 0
    
    print(f"\n📋 Epoch range: {actual_start_epoch} to {final_epoch - 1}")
    
    for i in range(actual_start_epoch, final_epoch):
        try:
            print(f"\n{'='*70}")
            print(f"🔄 Epoch {i} / {final_epoch - 1}")
            print(f"{'='*70}")
        
            # Shuffle dataset to ensure variety
            print(f"📥 Loading dataset page {i}...")
            df_train = read_data(dataset_file, i, 200000)
            df_train = df_train.sample(frac=1, random_state=i)  # Shuffle
            print(f"✅ Loaded {len(df_train)} samples")
        
            # Create dataloaders
            print(f"🔨 Creating dataloaders...")
            (dataloader, dataloader_test) = create_dataLoaders(df_train)
        
            # Train model
            print(f"🎓 Training...")
            train(model, dataloader, dataloader_test, device)
        
            # Save model after each iteration
            checkpoint_path = f'{model_name}_epoch{i}.pth'
            torch.save(model.state_dict(), checkpoint_path)
            print(f"💾 Checkpoint saved: {checkpoint_path}")
            epoch_count += 1
            
        except Exception as e:
            print(f"❌ Error in epoch {i}: {type(e).__name__}: {e}")
            failed_epochs.append((i, str(e)))
            print(f"   Skipping epoch {i} and continuing...")
            continue
    
    # Resumen de entrenamiento
    print(f"\n{'='*70}")
    print(f"🏁 TRAINING SUMMARY")
    print(f"{'='*70}")
    print(f"✅ Completed epochs: {epoch_count} / {num_epochs}")
    print(f"⚠️  Failed epochs: {len(failed_epochs)}")
    if failed_epochs:
        for epoch, error in failed_epochs:
            print(f"   - Epoch {epoch}: {error}")
    
    if epoch_count == 0:
        print("❌ No epochs completed successfully. Check your dataset or configuration.")
    else:
        print(f"✅ Training completed: {epoch_count}/{num_epochs} epochs!")

In [10]:
import random

def test_model_quality(model, file_path, device, n_samples=100):
    """Test model accuracy on a sample of the dataset"""
    model.eval()
    
    try:
        # 1. Cargar el dataset y extraer muestra aleatoria
        print(f"📥 Loading test data from {file_path}...")
        df = read_data(file_path, 1, 2000)
        
        if len(df) == 0:
            print("❌ Dataset is empty!")
            return
        
        # Ajustar n_samples al tamaño del dataset
        n_samples = min(n_samples, len(df))
        df_sample = df.sample(n=n_samples).reset_index(drop=True)
        
        top1_hits = 0
        top5_hits = 0
        total = 0
        errors = 0
        
        print(f"🧪 Testing on {n_samples} positions...")
        
        with torch.no_grad():
            for i, row in df_sample.iterrows():
                try:
                    # Preprocesar la entrada
                    input_tensor = torch.tensor(row['board'], dtype=torch.float32).unsqueeze(0).to(device)
                    target_idx = int(row['best'])
                    
                    # Predicción
                    policy_logits, value = model(input_tensor)
                    
                    # Calcular Top-K
                    _, top5_indices = torch.topk(policy_logits, 5, dim=1)
                    top5_indices = top5_indices.squeeze()
                    
                    # Manejar caso escalar (si batch_size=1)
                    if top5_indices.dim() == 0:
                        top5_indices = top5_indices.unsqueeze(0)
                    
                    top5_list = top5_indices.cpu().numpy().tolist()
                    
                    # Asegurar que es una lista
                    if not isinstance(top5_list, list):
                        top5_list = [top5_list]
                    
                    # Comprobación Top-1
                    if len(top5_list) > 0 and top5_list[0] == target_idx:
                        top1_hits += 1
                    
                    # Comprobación Top-5
                    if target_idx in top5_list:
                        top5_hits += 1
                    
                    total += 1
                    
                except Exception as e:
                    errors += 1
                    print(f"  ⚠️ Error processing sample {i}: {e}")
                    continue
        
        # Resultados finales
        if total == 0:
            print("❌ No samples processed successfully!")
            return
        
        top1_acc = (top1_hits / total) * 100
        top5_acc = (top5_hits / total) * 100
        
        print("\n" + "="*40)
        print(f"📊 TEST RESULTS (n={total})")
        print("="*40)
        print(f"✅ Top-1 Accuracy: {top1_acc:.1f}%  (Exact move)")
        print(f"🎯 Top-5 Accuracy: {top5_acc:.1f}%  (In top 5)")
        if errors > 0:
            print(f"⚠️  Processing errors: {errors}")
        print("="*40)
        
    except Exception as e:
        print(f"❌ Test failed: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

# Para llamarlo:
#test_model_quality(model, 'lc0_converted.parquet.gzip', device)

## 5️⃣ Model Testing & Evaluation
Functions to evaluate model performance on unseen data with Top-1 and Top-5 accuracy metrics.

## 6️⃣ Interactive Training Form
Run the cell below to start an interactive configuration form where you can:
- Choose your dataset
- Set the trained model name
- Load a base model (optional, for transfer learning)
- Configure training parameters

In [ ]:
# ============================================================================
# 🎯 MAIN EXECUTION - Interactive Configuration Form
# ============================================================================

def get_available_files(extension='.pth'):
    """Get list of available files with given extension in current directory"""
    try:
        return [f for f in os.listdir() if f.endswith(extension)]
    except Exception as e:
        print(f"⚠️ Could not list files: {e}")
        return []

def print_config_form():
    """Display the training configuration form with available options"""
    print("\n" + "="*70)
    print("🎮 CHESS TRAINING CONFIGURATION FORM")
    print("="*70)
    
    # Dataset options
    print("\n📊 AVAILABLE DATASETS:")
    datasets = get_available_files('.parquet.gzip') + get_available_files('.parquet')
    if datasets:
        for idx, ds in enumerate(datasets, 1):
            print(f"  [{idx}] {ds}")
    else:
        print("  ❌ No parquet files found. Using 'lc0_converted.parquet.gzip'")
        datasets = ['lc0_converted.parquet.gzip']
    
    # Model name input
    print("\n💾 MODEL NAME:")
    print("  Enter a name for your trained model (without extension)")
    print("  Example: 'chessmarro_v2', 'mymodel', etc.")
    
    # Base model options
    print("\n🔧 BASE MODEL (Optional):")
    base_models = get_available_files('.pth')
    if base_models:
        print("  [0] None (Train from scratch)")
        for idx, bm in enumerate(base_models, 1):
            print(f"  [{idx}] {bm}")
    else:
        print("  [0] None (Train from scratch)")
        print("  ❌ No .pth files found")
    
    print("\n" + "="*70)

def validate_model_name(name):
    """Validate model name"""
    if not name or not isinstance(name, str):
        return False, "Model name must be a non-empty string"
    if len(name) > 100:
        return False, "Model name too long (max 100 chars)"
    if any(c in name for c in ['/', '\\', '\0']):
        return False, "Model name contains invalid characters"
    return True, "OK"

def main_training_form():
    """Interactive form to configure and start training"""
    
    print_config_form()
    
    # Dataset selection
    datasets = get_available_files('.parquet.gzip') + get_available_files('.parquet')
    if not datasets:
        datasets = ['lc0_converted.parquet.gzip']
    
    print("\nSelect dataset [1]: ", end="")
    dataset_choice = input().strip() or "1"
    try:
        dataset_idx = int(dataset_choice) - 1
        dataset_file = datasets[dataset_idx] if 0 <= dataset_idx < len(datasets) else datasets[0]
    except (ValueError, IndexError):
        dataset_file = datasets[0]
        print(f"  ⚠️ Invalid selection. Using: {dataset_file}")
    
    # Validar que el dataset existe
    if not os.path.exists(dataset_file):
        print(f"❌ Dataset file not found: {dataset_file}")
        return None, None, None
    
    # Model name input with validation
    is_valid = False
    model_name = None
    while not is_valid:
        print("\nEnter model name [chessmarro]: ", end="")
        model_name = input().strip() or "chessmarro"
        is_valid, msg = validate_model_name(model_name)
        if not is_valid:
            print(f"  ❌ {msg}. Try again.")
    
    # Base model selection
    base_models = get_available_files('.pth')
    base_model_path = None
    
    if base_models:
        print("\nSelect base model [0]: ", end="")
        base_choice = input().strip() or "0"
        try:
            base_idx = int(base_choice) - 1
            if base_idx >= 0 and base_idx < len(base_models):
                base_model_path = base_models[base_idx]
                # Validar que el archivo existe
                if not os.path.exists(base_model_path):
                    print(f"  ⚠️ Base model file not found: {base_model_path}")
                    base_model_path = None
        except (ValueError, IndexError):
            pass
    
    # Additional parameters
    try:
        max_epochs = get_total_pages(dataset_file, 200000)
        max_epochs = max(1, max_epochs)  # Al menos 1
    except Exception as e:
        print(f"⚠️ Could not determine dataset size: {e}")
        max_epochs = 7
    
    # Number of epochs
    num_epochs = None
    while num_epochs is None:
        print(f"\nNumber of epochs [{max_epochs}]: ", end="")
        num_epochs_input = input().strip() or str(max_epochs)
        try:
            num_epochs = int(num_epochs_input)
            if num_epochs < 1:
                print("  ❌ Number of epochs must be >= 1")
                num_epochs = None
        except ValueError:
            print("  ❌ Invalid input. Enter a number.")
            num_epochs = None
    
    # Start epoch
    start_epoch = None
    while start_epoch is None:
        print("\nStart epoch [0]: ", end="")
        start_epoch_input = input().strip() or "0"
        try:
            start_epoch = int(start_epoch_input)
            if start_epoch < 0:
                print("  ❌ Start epoch must be >= 0")
                start_epoch = None
        except ValueError:
            print("  ❌ Invalid input. Enter a number.")
            start_epoch = None
    
    # Confirmation
    print("\n" + "="*70)
    print("⚙️  TRAINING CONFIGURATION:")
    print(f"  📊 Dataset: {dataset_file} ({os.path.getsize(dataset_file) / (1024**3):.2f} GB)")
    print(f"  💾 Model Name: {model_name}")
    print(f"  🔧 Base Model: {base_model_path if base_model_path else 'None (train from scratch)'}")
    print(f"  📈 Total Epochs: {num_epochs}")
    print(f"  🔄 Start Epoch: {start_epoch}")
    print("="*70)
    print("\nProceed with training? (yes/no) [yes]: ", end="")
    proceed = input().strip().lower() or "yes"
    
    if proceed in ['yes', 'y']:
        print("\n🚀 Starting training...\n")
        
        try:
            # Initialize model
            print("🏗️  Initializing model...")
            device, model = initialize_model(base_model_path)
            print(f"✅ Model initialized on {device}")
            
            # Start training loop
            start_training_loop(model, device, dataset_file, model_name, start_epoch, num_epochs)
            
            # Save final model
            final_model_path = f'{model_name}_final.pth'
            torch.save(model.state_dict(), final_model_path)
            print(f"\n✅ Training completed! Final model saved to: {final_model_path}")
            
            # Optional: Test the model
            print("\nTest trained model? (yes/no) [no]: ", end="")
            test_choice = input().strip().lower() or "no"
            if test_choice in ['yes', 'y']:
                try:
                    print("\n🧪 Testing model quality...")
                    test_model_quality(model, dataset_file, device, n_samples=100)
                except Exception as e:
                    print(f"⚠️ Testing failed: {e}")
            
            return model, device, model_name
            
        except Exception as e:
            print(f"\n❌ Training failed: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()
            return None, None, None
    else:
        print("❌ Training cancelled.")
        return None, None, None

# Run the main form
if __name__ == "__main__":
    model, device, model_name = main_training_form()


🎮 CHESS TRAINING CONFIGURATION FORM

📊 AVAILABLE DATASETS:
  [1] lc0_converted.parquet.gzip

💾 MODEL NAME:
  Enter a name for your trained model (without extension)
  Example: 'chessmarro_v2', 'mymodel', etc.

🔧 BASE MODEL (Optional):
  [0] None (Train from scratch)
  [1] chessmarro_final.pth
  [2] chessmarro_epoch0.pth
  [3] chessmarro_v9_final.pth
  [4] chessmarro_epoch1.pth


Select dataset [1]: 


Enter model name [chessmarro]: 


Select base model [0]: 


Number of epochs [1]: 


Start epoch [0]: 

...

   Avg Train Loss: 4.9218
🔎 Test Loss: 4.1624 | Acc: 42.31% | MSE: 0.1455


Training Epoch 5: 100%|██████████| 3/3 [00:00<00:00, 62.07it/s, L_pol=3.12, L_val=0.133]


   Avg Train Loss: 4.6157
🔎 Test Loss: 4.1844 | Acc: 42.31% | MSE: 0.1369


Training Epoch 6: 100%|██████████| 3/3 [00:00<00:00, 61.22it/s, L_pol=3.12, L_val=0.154]


   Avg Train Loss: 4.7862
🔎 Test Loss: 4.2082 | Acc: 42.31% | MSE: 0.1442
⚠️ Early stopping triggered (no improvement).
🎯 Training completed. Best accuracy: 42.31%
💾 Checkpoint saved: chessmarro_epoch2.pth

🏁 TRAINING SUMMARY
✅ Completed epochs: 1 / 1
⚠️  Failed epochs: 0
✅ Training completed: 1/1 epochs!

✅ Training completed! Final model saved to: chessmarro_final.pth

Test trained model? (yes/no) [no]: 